## Stats relation


In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv(r"..\combined_data.csv")

In [2]:
teams = pd.unique(pd.concat([df["HomeTeam"], df["AwayTeam"]]))
print("Unique teams:", len(teams))
print(sorted(teams))

Unique teams: 24
['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton', 'Burnley', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Leeds', 'Leicester', 'Liverpool', 'Man City', 'Man United', 'Newcastle', 'Norwich', 'Sheffield United', 'Southampton', 'Tottenham', 'Watford', 'West Brom', 'West Ham', 'Wolves']


### baseline result rate(%)

In [3]:
baseline = (df["FTR"].value_counts(normalize=True) * 100).round(1)
print(baseline.reindex(["H", "D", "A"]))

FTR
H    41.0
D    23.2
A    35.8
Name: proportion, dtype: float64


In [4]:
home = (
    df.groupby("HomeTeam")["FTR"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reindex(columns=["H", "D", "A"], fill_value=0)
    .mul(100)
    .round(3)
    .rename(columns={"H": "win_pct", "D": "draw_pct", "A": "lose_pct"})
)
home["n"] = df.groupby("HomeTeam").size()

away = (
    df.groupby("AwayTeam")["FTR"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reindex(columns=["H", "D", "A"], fill_value=0)
    .mul(100)
    .round(3)
    .rename(columns={"A": "win_pct", "D": "draw_pct", "H": "lose_pct"})
)
away["n"] = df.groupby("AwayTeam").size()

profiles = home.add_suffix("_home").join(away.add_suffix("_away"), how="outer")

print("As HOME:")
print(home.sort_values("win_pct", ascending=False))
print("\nAs AWAY:")
print(away.sort_values("win_pct", ascending=False))
print("\nJoined profiles (all teams):")
print(profiles.sort_index())
print("\nRows:", len(home), len(away), len(profiles))

As HOME:
FTR               win_pct  draw_pct  lose_pct   n
HomeTeam                                         
Liverpool          74.510    13.725    11.765  51
Man City           72.549    11.765    15.686  51
Tottenham          60.784    11.765    27.451  51
Leicester          52.941    15.686    31.373  51
Arsenal            50.980    21.569    27.451  51
Man United         49.020    27.451    23.529  51
Chelsea            45.098    31.373    23.529  51
Everton            43.137    17.647    39.216  51
West Ham           43.137    23.529    33.333  51
Newcastle          37.255    31.373    31.373  51
Brentford          36.842    15.789    47.368  19
Wolves             35.294    25.490    39.216  51
Aston Villa        35.294    21.569    43.137  51
Sheffield United   34.375     9.375    56.250  32
Southampton        33.333    23.529    43.137  51
Crystal Palace     33.333    33.333    33.333  51
Leeds              31.579    28.947    39.474  38
Bournemouth        30.769    30.769    38

In [5]:
profiles.to_csv("team_home_away_profiles.csv", index_label="Team")
print("Saved team_home_away_profiles.csv")

Saved team_home_away_profiles.csv


In [6]:
# 1. Select Home and Away teams
team_a = "Arsenal"   # home
team_b = "Chelsea"   # away
# 2. Matchup History
matchup = df[(df["HomeTeam"] == team_a) & (df["AwayTeam"] == team_b)]
n = len(matchup)
outcome = (
    matchup["FTR"]
    .value_counts(normalize=True)
    .reindex(["H", "D", "A"])
    .fillna(0)
    * 100
)
print(f"Historical Head-to-Head Matches: n = {n}")
display(outcome.round(1).to_frame(name="Historical Win %"))
# 3. Calculate Referee Card Statistics from df
lg_hy = df['HY'].mean()
lg_ay = df['AY'].mean()
lg_hr = df['HR'].mean()
lg_ar = df['AR'].mean()
ref_cards = df.groupby('Referee').agg(
    n=('FTR', 'count'),
    total_yellow=('HY', lambda x: x.sum() + df.loc[x.index, 'AY'].sum()),
    total_red=('HR', lambda x: x.sum() + df.loc[x.index, 'AR'].sum()),
    home_yellow=('HY', 'mean'),
    away_yellow=('AY', 'mean'),
    home_red=('HR', 'mean'),
    away_red=('AR', 'mean')
).reset_index()
ref_cards['yellows_per_game'] = (ref_cards['total_yellow'] / ref_cards['n']).round(2)
ref_cards['reds_per_game'] = (ref_cards['total_red'] / ref_cards['n']).round(3)
ref_cards['yellow_gap'] = ((ref_cards['home_yellow'] - ref_cards['away_yellow']) - (lg_hy - lg_ay)).round(3)
ref_cards['red_gap'] = ((ref_cards['home_red'] - ref_cards['away_red']) - (lg_hr - lg_ar)).round(3)
# 4. Color styling (Green for positive, Red for negative)
def color_gap(v):
    if pd.isna(v):
        return ""
    if v < 0:
        return "color: red; font-weight: bold;"
    if v > 0:
        return "color: green; font-weight: bold;"
    return ""
try:
    styled_ref_cards = ref_cards.style.map(
        color_gap,
        subset=["yellow_gap", "red_gap"]
    )
except AttributeError:
    styled_ref_cards = ref_cards.style.applymap(
        color_gap,
        subset=["yellow_gap", "red_gap"]
    )
display(styled_ref_cards)


Historical Head-to-Head Matches: n = 3


,Historical Win %
FTR,
H,33.3
D,0.0
A,66.7


,Referee,n,total_yellow,total_red,home_yellow,away_yellow,home_red,away_red,yellows_per_game,reds_per_game,yellow_gap,red_gap
0,A Madley,40,107,4,1.225000,1.450000,0.025000,0.075000,2.680000,0.100000,-0.138000,-0.036000
1,A Marriner,59,157,5,1.169492,1.491525,0.050847,0.033898,2.660000,0.085000,-0.235000,0.031000
2,A Moss,1,4,0,1.000000,3.000000,0.000000,0.000000,4.000000,0.000000,-1.913000,0.014000
3,A Taylor,77,263,12,1.714286,1.701299,0.103896,0.051948,3.420000,0.156000,0.100000,0.066000
4,C Kavanagh,58,186,5,1.431034,1.775862,0.068966,0.017241,3.210000,0.086000,-0.258000,0.065000
5,C Pawson,62,239,7,2.048387,1.806452,0.048387,0.064516,3.850000,0.113000,0.329000,-0.002000
6,D Coote,52,191,6,1.788462,1.884615,0.057692,0.057692,3.670000,0.115000,-0.009000,0.014000
7,D England,27,96,1,1.555556,2.000000,0.000000,0.037037,3.560000,0.037000,-0.357000,-0.023000
8,G Scott,37,106,8,1.405405,1.459459,0.108108,0.108108,2.860000,0.216000,0.033000,0.014000
9,J Brooks,4,21,0,2.750000,2.500000,0.000000,0.000000,5.250000,0.000000,0.337000,0.014000


In [7]:
df = df.copy()
df["yellows"] = df["HY"] + df["AY"]
df["reds"] = df["HR"] + df["AR"]

league_y = df["yellows"].mean()
league_r = df["reds"].mean()

ref_cards = (
    df.groupby("Referee")
    .agg(
        n=("FTR", "size"),
        total_yellow=("yellows", "sum"),
        total_red=("reds", "sum"),
        yellows_per_game=("yellows", "mean"),
        reds_per_game=("reds", "mean"),
        home_yellow=("HY", "mean"),
        away_yellow=("AY", "mean"),
        home_red=("HR", "mean"),
        away_red=("AR", "mean"),
    )
)
ref_cards["yellow_gap"] = ref_cards["yellows_per_game"] - league_y
ref_cards["red_gap"] = ref_cards["reds_per_game"] - league_r
ref_cards = ref_cards.sort_values("n", ascending=False)

ref_cards["n"] = ref_cards["n"].astype(int)
ref_cards["total_yellow"] = ref_cards["total_yellow"].astype(int)
ref_cards["total_red"] = ref_cards["total_red"].astype(int)

print(f"league yellows/game = {league_y:.2f}, reds/game = {league_r:.3f}")

def color_gap(v):
    if pd.isna(v):
        return ""
    if v < 0:
        return "color: red"
    if v > 0:
        return "color: green"
    return ""

(
    ref_cards.style
    .format({
        "yellows_per_game": "{:.2f}",
        "home_yellow": "{:.2f}",
        "away_yellow": "{:.2f}",
        "yellow_gap": "{:.2f}",
        "reds_per_game": "{:.3f}",
        "home_red": "{:.3f}",
        "away_red": "{:.3f}",
        "red_gap": "{:.3f}",
    })
    .map(color_gap, subset=["yellow_gap", "red_gap"])
)

league yellows/game = 3.22, reds/game = 0.120


,n,total_yellow,total_red,yellows_per_game,reds_per_game,home_yellow,away_yellow,home_red,away_red,yellow_gap,red_gap
Referee,,,,,,,,,,,
A Taylor,77,263,12,3.42,0.156,1.71,1.70,0.104,0.052,0.20,0.036
M Oliver,74,220,10,2.97,0.135,1.49,1.49,0.054,0.081,-0.25,0.016
M Atkinson,74,190,7,2.57,0.095,1.26,1.31,0.027,0.068,-0.65,-0.025
M Dean,71,258,12,3.63,0.169,1.92,1.72,0.056,0.113,0.42,0.049
J Moss,67,182,8,2.72,0.119,1.40,1.31,0.015,0.104,-0.50,-0.000
P Tierney,66,241,11,3.65,0.167,1.74,1.91,0.076,0.091,0.43,0.047
K Friend,62,198,8,3.19,0.129,1.71,1.48,0.048,0.081,-0.03,0.009
C Pawson,62,239,7,3.85,0.113,2.05,1.81,0.048,0.065,0.64,-0.007
A Marriner,59,157,5,2.66,0.085,1.17,1.49,0.051,0.034,-0.56,-0.035


In [8]:
ref_cards.to_csv("ref_cards.csv", index_label="Referee")
print("Saved ref_cards.csv")

Saved ref_cards.csv


### Team Attack & Defense Classification Logic

#### 1. League Baseline Goals
* **League Avg Home Goals (`league_home_goals_avg`)** = Average goals scored by home teams across the league (~1.445)
* **League Avg Away Goals (`league_away_goals_avg`)** = Average goals scored by away teams across the league (~1.312)

---

#### 2. Strength Ratio Calculations
* **`home_attack`** = `(Team Home Goals Scored Avg) / league_home_goals_avg`
* **`home_defense`** = `(Team Home Goals Conceded Avg) / league_away_goals_avg`
* **`away_attack`** = `(Team Away Goals Scored Avg) / league_away_goals_avg`
* **`away_defense`** = `(Team Away Goals Conceded Avg) / league_home_goals_avg`

---

#### 3. Classification Logic (`home_type` & `away_type`)
*A ratio of 1.0 represents the league average.*
* **Strong Attack & Defense**: `attack >= 1.0` AND `defense <= 1.0` (scores above average & concedes below average)
* **Attack Team**: `attack >= 1.0` (scores above league average)
* **Defense Team**: `defense < 1.0` (concedes fewer goals than league average)
* **Weak Both**: `attack < 1.0` AND `defense > 1.0` (below average in scoring & conceding)

---

#### 4. Overall Classification (`overall_type`)
* **Average Attack Power** = `(home_attack + away_attack) / 2`
* **Average Defense Power** = `2.0 - ((home_defense + away_defense) / 2)` *(lower goals conceded = stronger defense)*
* **Result**:
  * **Attack Team**: if `Average Attack Power >= Average Defense Power`
  * **Defense Team**: if `Average Defense Power > Average Attack Power`


In [9]:
import numpy as np
import pandas as pd

# 1. League Baseline Goals
league_home_goals_avg = df['FTHG'].mean()
league_away_goals_avg = df['FTAG'].mean()

print(f"League Avg Home Goals: {league_home_goals_avg:.3f}")
print(f"League Avg Away Goals: {league_away_goals_avg:.3f}\n")

# 2. Calculate Home Stats
home_stats = df.groupby('HomeTeam').agg(
    home_goals_scored=('FTHG', 'mean'),
    home_goals_conceded=('FTAG', 'mean')
).reset_index()

home_stats['home_attack_val'] = home_stats['home_goals_scored'] / league_home_goals_avg
home_stats['home_defense_val'] = home_stats['home_goals_conceded'] / league_away_goals_avg

# 3. Calculate Away Stats
away_stats = df.groupby('AwayTeam').agg(
    away_goals_scored=('FTAG', 'mean'),
    away_goals_conceded=('FTHG', 'mean')
).reset_index()

away_stats['away_attack_val'] = away_stats['away_goals_scored'] / league_away_goals_avg
away_stats['away_defense_val'] = away_stats['away_goals_conceded'] / league_home_goals_avg

# 4. Merge into one table
team_type_df = pd.merge(
    home_stats, 
    away_stats, 
    left_on='HomeTeam', 
    right_on='AwayTeam'
).drop(columns=['AwayTeam']).rename(columns={'HomeTeam': 'team'})

# 5. Classifications
def get_home_type(row):
    if row['home_attack_val'] >= 1.0 and row['home_defense_val'] <= 1.0:
        return "Strong Attack & Defense"
    elif row['home_attack_val'] >= 1.0:
        return "Attack Team"
    elif row['home_defense_val'] < 1.0:
        return "Defense Team"
    else:
        return "Weak Both"

def get_away_type(row):
    if row['away_attack_val'] >= 1.0 and row['away_defense_val'] <= 1.0:
        return "Strong Attack & Defense"
    elif row['away_attack_val'] >= 1.0:
        return "Attack Team"
    elif row['away_defense_val'] < 1.0:
        return "Defense Team"
    else:
        return "Weak Both"

def get_overall_type(row):
    avg_attack = (row['home_attack_val'] + row['away_attack_val']) / 2.0
    avg_defense_power = 2.0 - ((row['home_defense_val'] + row['away_defense_val']) / 2.0)
    return "Attack Team" if avg_attack >= avg_defense_power else "Defense Team"

team_type_df['home_type'] = team_type_df.apply(get_home_type, axis=1)
team_type_df['away_type'] = team_type_df.apply(get_away_type, axis=1)
team_type_df['overall_type'] = team_type_df.apply(get_overall_type, axis=1)

# 6. Format Attack & Defense as +/- percentages
def format_pct_shift(val):
    pct = (val - 1.0) * 100.0
    return f"{pct:+.3f}"

team_type_df['home_attack%'] = team_type_df['home_attack_val'].apply(format_pct_shift)
team_type_df['home_defense%'] = team_type_df['home_defense_val'].apply(format_pct_shift)
team_type_df['away_attack%'] = team_type_df['away_attack_val'].apply(format_pct_shift)
team_type_df['away_defense%'] = team_type_df['away_defense_val'].apply(format_pct_shift)

# Reorder columns cleanly
team_type_df = team_type_df[[
    'team',
    'overall_type',
    'home_type',
    'away_type',
    'home_attack%',
    'home_defense%',
    'away_attack%',
    'away_defense%'
]]

# Save to CSV
team_type_df.to_csv("team_attack_or_defense_type.csv", index=False)

# Display table
display(team_type_df)


League Avg Home Goals: 1.445
League Avg Away Goals: 1.312



,team,overall_type,home_type,away_type,home_attack%,home_defense%,away_attack%,away_defense%
0,Arsenal,Defense Team,Strong Attack & Defense,Strong Attack & Defense,+9.905,-16.293,+6.129,-11.805
1,Aston Villa,Attack Team,Attack Team,Defense Team,+3.121,+19.581,-2.840,-5.020
2,Bournemouth,Defense Team,Weak Both,Weak Both,-20.154,+11.418,-35.495,+11.784
3,Brentford,Defense Team,Defense Team,Attack Team,-19.874,-15.742,+4.319,+27.473
4,Brighton,Defense Team,Defense Team,Defense Team,-21.303,-11.809,-17.788,-6.377
5,Burnley,Defense Team,Weak Both,Weak Both,-34.871,+4.634,-26.756,+4.478
6,Chelsea,Attack Team,Strong Attack & Defense,Strong Attack & Defense,+13.976,-19.283,+39.013,-30.801
7,Crystal Palace,Defense Team,Defense Team,Weak Both,-22.659,-7.324,-14.798,+11.262
8,Everton,Defense Team,Weak Both,Weak Both,-6.377,+0.149,-17.788,+15.332
9,Fulham,Defense Team,Weak Both,Defense Team,-67.221,+12.344,-27.779,-8.948
